*  DSC 530 Data Exploration and Analysis
*  Weeks 5 & 6 Coding Assignment
*  Adam Luna

## Setup

In this section, I import the libraries I will use throughout the assignment and define file paths
to the Chapter 4 exercise datasets. I use `Path` so the notebook can reliably find the data files
inside the `ch_04/exercises/` folder.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# This notebook is stored in:
# Hands-On-Data-Analysis-with-Pandas-2nd-edition-master/assignments/
# So the repo root is one folder up from the current working directory.
REPO_ROOT = Path.cwd().parent

# All datasets for this assignment are confirmed to be in ch_04/exercises/
CH04_EXERCISES = REPO_ROOT / "ch_04" / "exercises"

EARTHQUAKES_CSV = CH04_EXERCISES / "earthquakes.csv"
FAANG_CSV = CH04_EXERCISES / "faang.csv"
COVID_CSV = CH04_EXERCISES / "covid19_cases.csv"

# Display paths so it's clear what files the notebook expects
EARTHQUAKES_CSV, FAANG_CSV, COVID_CSV

(PosixPath('/Users/adamluna/dsc530/miniconda3/envs/book_env/Hands-On-Data-Analysis-with-Pandas-2nd-edition-master/ch_04/exercises/earthquakes.csv'),
 PosixPath('/Users/adamluna/dsc530/miniconda3/envs/book_env/Hands-On-Data-Analysis-with-Pandas-2nd-edition-master/ch_04/exercises/faang.csv'),
 PosixPath('/Users/adamluna/dsc530/miniconda3/envs/book_env/Hands-On-Data-Analysis-with-Pandas-2nd-edition-master/ch_04/exercises/covid19_cases.csv'))

# Hands-on data analysis with Pandas

# Chapter 4, Exercise 1

## Exercise 1: Japan earthquakes (mb) with magnitude ≥ 4.9

In this exercise, I load the earthquake dataset and filter it down to earthquakes in Japan that
have a magnitude of 4.9 or greater using the `mb` magnitude type. This produces a subset that
matches the criteria in the prompt.

In [2]:
# Read the earthquake data
eq = pd.read_csv(EARTHQUAKES_CSV)

# Filter: Japan + magType mb + magnitude >= 4.9
eq_japan_mb_49 = eq[
    eq["place"].str.contains("Japan", na=False)
    & (eq["magType"] == "mb")
    & (eq["mag"] >= 4.9)
]

# Display the number of rows and a preview of results
eq_japan_mb_49.shape, eq_japan_mb_49.head()

((4, 6),
       mag magType           time                         place  tsunami  \
 1563  4.9      mb  1538977532250  293km ESE of Iwo Jima, Japan        0   
 2576  5.4      mb  1538697528010    37km E of Tomakomai, Japan        0   
 3072  4.9      mb  1538579732490     15km ENE of Hasaki, Japan        0   
 3632  4.9      mb  1538450871260    53km ESE of Hitachi, Japan        0   
 
      parsed_place  
 1563        Japan  
 2576        Japan  
 3072        Japan  
 3632        Japan  )

## Chapter 4, Exercise 2

## Exercise 2: Magnitude bins (ml) and counts per bin

In this exercise, I use the `ml` magnitude type and create bins for each full number of earthquake
magnitude. For example, values between 0 and 1 fall into (0, 1], values between 1 and 2 fall into (1, 2], and so on.
After creating the bins, I count how many earthquakes fall into each one.

In [3]:
# Filter to ml magnitude type and keep valid magnitude values
eq_ml = eq[(eq["magType"] == "ml") & eq["mag"].notna()].copy()

# The prompt examples start at (0,1], so I keep magnitudes > 0
eq_ml = eq_ml[eq_ml["mag"] > 0]

# Create bin edges from 0 up to the next whole number above the max magnitude
max_mag = eq_ml["mag"].max()
upper_edge = int(np.ceil(max_mag))
bin_edges = np.arange(0, upper_edge + 1, 1)

# Bin the magnitudes into (0,1], (1,2], ...
eq_ml["mag_bin"] = pd.cut(eq_ml["mag"], bins=bin_edges, right=True)

# Count how many are in each bin and display in ascending bin order
bin_counts = eq_ml["mag_bin"].value_counts().sort_index()
bin_counts

mag_bin
(0, 1]    2207
(1, 2]    3105
(2, 3]     862
(3, 4]     122
(4, 5]       2
(5, 6]       1
Name: count, dtype: int64

## Chapter 4, Exercise 4

## Exercise 4: Crosstab of tsunami vs magType (maximum magnitude)

In this exercise, I build a crosstab between the `tsunami` column and `magType`. Instead of showing
frequency counts, I show the maximum observed magnitude (`mag`) for each combination. I also place
the magnitude type along the columns, as required.

In [4]:
# Crosstab where each cell contains the max magnitude for that (tsunami, magType) combination
tsunami_magtype_maxmag = pd.crosstab(
    index=eq["tsunami"],
    columns=eq["magType"],
    values=eq["mag"],
    aggfunc="max"
)

tsunami_magtype_maxmag

magType,mb,mb_lg,md,mh,ml,ms_20,mw,mwb,mwr,mww
tsunami,,,,,,,,,,
0,5.6,3.5,4.11,1.1,4.2,NaN,3.83,5.8,4.8,6.0
1,6.1,NaN,NaN,NaN,5.1,5.7,4.41,NaN,NaN,7.5


## Chapter 4, Exercise 6

## Exercise 6: FAANG pivot table comparing average OHLC and volume

In this exercise, I load the FAANG dataset and create a pivot table to compare the stocks.
I put `ticker` in the rows and calculate the average of OHLC (`open`, `high`, `low`, `close`)
and `volume` for each ticker.

In [5]:
# Read the FAANG dataset
faang = pd.read_csv(FAANG_CSV)

# Create a pivot table: ticker as rows, averages of OHLC + volume as values
faang_pivot = faang.pivot_table(
    index="ticker",
    values=["open", "high", "low", "close", "volume"],
    aggfunc="mean"
)

faang_pivot

,close,high,low,open,volume
ticker,,,,,
AAPL,47.263357,47.748526,46.795877,47.277859,1.360803e+08
AMZN,1641.726176,1662.839839,1619.840519,1644.072709,5.648994e+06
FB,171.510956,173.613347,169.303148,171.472948,2.765860e+07
GOOG,1113.225134,1125.777606,1101.001658,1113.554101,1.741965e+06
NFLX,319.290319,325.219322,313.187330,319.620558,1.146962e+07


## Chapter 4, Exercise 7

## Exercise 7: Z-scores for Amazon (AMZN) numeric columns in Q4 2018

In this exercise, I filter the FAANG dataset down to Amazon (ticker `AMZN`) during Q4 2018
(October through December). Then I calculate z-scores for each numeric column using `apply()`.
This standardizes each column so values represent how many standard deviations they are from the mean.

In [6]:
# Ensure the date column is treated as datetime so filtering by date is reliable
faang["date"] = pd.to_datetime(faang["date"])

# Filter to AMZN in Q4 2018 (Oct 1 through Dec 31)
amzn_q4_2018 = faang[
    (faang["ticker"] == "AMZN")
    & (faang["date"] >= "2018-10-01")
    & (faang["date"] <= "2018-12-31")
].copy()

# Select numeric columns only (OHLC and volume should be numeric)
numeric_cols = amzn_q4_2018.select_dtypes(include="number")

# Calculate z-scores with apply():
# z = (x - mean) / std
# Using ddof=0 is a common convention for z-scores
z_scores = numeric_cols.apply(lambda col: (col - col.mean()) / col.std(ddof=0))

# Display a preview
z_scores.head()

,high,low,open,close,volume
690,2.387026,2.522211,2.356591,2.405011,-1.643506
691,2.245193,2.265485,2.208392,2.172347,-0.868802
692,2.075493,2.157176,2.085185,2.041758,-0.927738
693,1.834088,1.795871,1.864908,1.736654,-0.127599
694,1.641251,1.566901,1.656015,1.597477,-0.301170


## Chapter 4, Exercise 10

## Exercise 10, Part A: Prepare the COVID-19 data

In this part, I load the COVID-19 dataset and prepare it for analysis by creating a datetime
index, cleaning country names, and sorting the data by date. These steps ensure the dataset
is ready for time-based analysis and comparisons across countries.

In [7]:
# Read the COVID-19 dataset
covid = pd.read_csv(COVID_CSV)

# Create a date column from dateRep (this dataset uses day/month/year formatting)
covid["date"] = pd.to_datetime(covid["dateRep"], dayfirst=True)

# Replace long country names with short versions across the dataframe
covid = covid.replace({
    "United_States_of_America": "USA",
    "United_Kingdom": "UK"
})

# Set date as the index and sort by date
covid = covid.set_index("date").sort_index()

# Preview to confirm the data is prepared correctly
covid.shape, covid.head()

((43718, 12),
                dateRep  day  month  year  cases  deaths  \
 date                                                      
 2019-12-31  31/12/2019   31     12  2019      0       0   
 2019-12-31  31/12/2019   31     12  2019      0       0   
 2019-12-31  31/12/2019   31     12  2019      0       0   
 2019-12-31  31/12/2019   31     12  2019      0       0   
 2019-12-31  31/12/2019   31     12  2019      0       0   
 
            countriesAndTerritories geoId countryterritoryCode  popData2019  \
 date                                                                         
 2019-12-31                 Belgium    BE                  BEL   11455519.0   
 2019-12-31                  Mexico    MX                  MEX  127575529.0   
 2019-12-31                 Ecuador    EC                  ECU   17373657.0   
 2019-12-31                  Russia    RU                  RUS  145872260.0   
 2019-12-31             Netherlands    NL                  NLD   17282163.0   
 
         

## Exercise 10, Part B: Identify the top five countries by cumulative cases

In this part, I calculate the total number of reported COVID-19 cases for each country and
identify the five countries with the highest cumulative case counts in this dataset snapshot.

In [8]:
# Calculate cumulative cases per country
cases_by_country = covid.groupby("countriesAndTerritories")["cases"].sum().sort_values(ascending=False)

# Select the top five countries by cumulative cases
top5_countries = cases_by_country.head(5)
top5_countries

countriesAndTerritories
USA       6724667
India     5308014
Brazil    4495183
Russia    1091186
Peru       756412
Name: cases, dtype: int64

## Exercise 10, Part C: Day with the largest number of cases (top five countries)

In this part, I focus on the five countries with the most cumulative cases and determine
the single day on which each country reported its highest number of new cases.

In [9]:
# Filter the dataset to the top five countries
top5_list = top5_countries.index.tolist()
covid_top5 = covid[covid["countriesAndTerritories"].isin(top5_list)].copy()

# For each country, find the maximum daily cases and the date it occurred
largest_day_top5 = covid_top5.groupby("countriesAndTerritories")["cases"].agg(
    max_cases="max",
    date_of_max=lambda s: s.idxmax()
)

largest_day_top5

,max_cases,date_of_max
countriesAndTerritories,,
Brazil,69074,2020-07-30
India,97894,2020-09-17
Peru,10143,2020-08-17
Russia,12640,2020-07-18
USA,78427,2020-07-25


## Exercise 10, Part D: 7-day average change in cases for the last week

In this part, I calculate the day-to-day change in reported COVID-19 cases for each of the
top five countries and compute a 7-day rolling average of that change. I then display the
results for the final week in the dataset.

In [10]:
# Pivot the data so dates are rows and countries are columns
cases_wide_top5 = covid_top5.pivot_table(
    index=covid_top5.index,
    columns="countriesAndTerritories",
    values="cases",
    aggfunc="sum"
).sort_index()

# Calculate daily change in cases
daily_change = cases_wide_top5.diff()

# Compute the 7-day rolling average of the daily change
rolling_7day_avg_change = daily_change.rolling(7).mean()

# Display the last week of results
rolling_7day_avg_change.tail(7)

countriesAndTerritories,Brazil,India,Peru,Russia,USA
date,,,,,
2020-09-13,479.285714,534.285714,-98.857143,40.428571,-474.285714
2020-09-14,35.285714,181.285714,73.142857,36.285714,473.714286
2020-09-15,697.428571,1142.857143,377.571429,46.285714,1513.000000
2020-09-16,3196.285714,59.571429,-65.000000,61.428571,3478.714286
2020-09-17,143.428571,308.428571,-29.428571,810.000000,-1047.000000
2020-09-18,-607.714286,-18.142857,-227.571429,-688.428571,865.714286
2020-09-19,-560.142857,-604.714286,-41.285714,57.285714,306.857143


## Exercise 10, Part E: First reported case date for each country (excluding China)

In this part, I determine the first date on which each country reported at least one COVID-19 case.
China is excluded as instructed, and the results are sorted chronologically.

In [11]:
# Keep only rows where cases are greater than zero
covid_nonzero = covid[covid["cases"] > 0].copy()

# Find the first date with reported cases for each country
first_case_date = covid_nonzero.groupby("countriesAndTerritories").apply(lambda g: g.index.min())

# Exclude China and sort by date
first_case_date_no_china = first_case_date[first_case_date.index != "China"].sort_values()

# Display a portion of the results
first_case_date_no_china.head(15), first_case_date_no_china.tail(15)

(countriesAndTerritories
 Thailand               2020-01-13
 Japan                  2020-01-15
 South_Korea            2020-01-20
 Taiwan                 2020-01-21
 USA                    2020-01-21
 Singapore              2020-01-24
 Vietnam                2020-01-24
 Malaysia               2020-01-25
 Nepal                  2020-01-25
 Australia              2020-01-25
 France                 2020-01-25
 Canada                 2020-01-26
 United_Arab_Emirates   2020-01-27
 Sri_Lanka              2020-01-28
 Germany                2020-01-28
 dtype: datetime64[ns],
 countriesAndTerritories
 Puerto_Rico                         2020-03-28
 Northern_Mariana_Islands            2020-03-31
 Botswana                            2020-04-01
 Burundi                             2020-04-01
 Sierra_Leone                        2020-04-01
 Bonaire, Saint Eustatius and Saba   2020-04-02
 Malawi                              2020-04-03
 Falkland_Islands_(Malvinas)         2020-04-04
 South_Sudan     

## Exercise 10, Part F: Percentile ranking of countries by cumulative cases

In this part, I rank countries based on their cumulative COVID-19 case totals using percentiles.
A higher percentile indicates that a country is closer to the top of the cumulative case
distribution.

In [12]:
# Calculate cumulative cases per country
cases_totals = covid.groupby("countriesAndTerritories")["cases"].sum()

# Compute percentile ranks (values between 0 and 1)
percentile_rank = cases_totals.rank(pct=True)

# Combine totals and percentile ranks into a single table
ranked_cases = pd.DataFrame({
    "cumulative_cases": cases_totals,
    "percentile_rank": percentile_rank
}).sort_values("cumulative_cases", ascending=False)

ranked_cases.head(15)

,cumulative_cases,percentile_rank
countriesAndTerritories,,
USA,6724667,1.000000
India,5308014,0.995238
Brazil,4495183,0.990476
Russia,1091186,0.985714
Peru,756412,0.980952
Colombia,750471,0.976190
Mexico,688954,0.971429
South_Africa,657627,0.966667
Spain,640040,0.961905
